# Day 2 Reference Solution: Employee, contractor & manager (HR)

**JUDGES ONLY. Never share with teams.**

This is the full registry the HR starter is asking teams to build toward: **four Kind 1 patterns** (Employee, Contractor, Manager, Case) and **four Kind 2 candidates deliberately rejected** (Floor, Grade, Amount, Appraisal), each with a reason.

It runs the public pack *and* the hidden pack. Because the starter's `grounded_answer` is fixed (record IDs in the context, the checked verdict beside each pair, `temperature=0`), the answer follows the graph verdict whenever retrieval returns the two records, so the **registry is what is being judged**.

The offline checker (`day2_offline_check.py --dataset HR`) reproduces the structural verdicts without Neo4j or an LLM.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [ ]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"
LLM_BASE_URL = None  # None = OpenAI. Otherwise your provider's OpenAI-compatible URL

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

### Connect to Neo4j Aura

Create a free [Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance if you haven't, then add `NEO4J_URI`, `NEO4J_USER` and `NEO4J_PASSWORD` to Colab's Secrets manager (padlock icon). If you hit `ServiceUnavailable`, the instance auto-paused: re-run this cell.

**Cannot use Neo4j?** Skip this cell. Replace the graph-building cell (section 7) and `graph_check` (section 8) with lookup-table versions that keep the same return shape. The Student Setup Guide shows the table structure.

In [ ]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

print(run_read("RETURN 'Connection successful' AS message")[0]["message"])

## 2. The dataset: campus operations & student services

179 records in 14 categories. Four pairs of categories deliberately share vocabulary (money, rooms, passes/permits, events). Most records stand alone; some name a person, a club or a ticket.

In [ ]:
DATASET_ID = "HR"
DATASET_NAME = "Employee, contractor & manager"

rec_categories = {
    'Recruitment': [
        'A job posting for a data analyst role was published on the careers page',
        'Recruiters screened a batch of applications for the sales team',
        'An interview panel was scheduled for a senior engineer candidate',
        'A candidate declined an offer after a counter from another company',
        'Reference checks were completed for a finance candidate',
        'The hiring freeze was lifted for two customer support roles',
        'A campus hiring drive was planned for the next graduating class',
        'An offer letter was revised after the salary band was updated',
        'A recruiter reopened a role that had stayed unfilled for two months',
        'A Grade L3 role was opened for a backend developer',
        'Hiring for the support team was paused until the Q3 appraisal closed',
        'Contractor K230 was engaged through a staffing agency for six months',
    ],
    'Onboarding': [
        'A new hire orientation was held for the incoming batch',
        'IT prepared laptops and access cards for the joiners',
        'A buddy was assigned to a new team member',
        'Onboarding paperwork was completed before the first day',
        'A welcome kit was sent to a remote joiner',
        'The induction schedule was shortened for experienced hires',
        "A new joiner's security training was marked pending",
        'Probation goals were shared with the new team members',
        'The onboarding survey showed a gap in first-week tool access',
        'Employee E1021 joined the finance team through an employee referral',
        'Contractor K214 was issued a laptop and access card before starting',
    ],
    'Payroll': [
        'Monthly payroll was processed and salary slips were released',
        'A tax declaration deadline was announced to all staff',
        'Payroll corrected an overpayment from the previous cycle',
        'A bonus payout was scheduled with the next salary run',
        'Provident fund contributions were reconciled by payroll',
        'Payroll closed the cycle after a final attendance check',
        'A salary advance request was approved by finance',
        'Reimbursement claims were added to the payroll file',
        'A bank file for salary transfers was validated',
        "Employee E1034's new salary was reflected in the payroll run",
        "Employee E1021's relocation allowance of 3000 dollars was approved",
    ],
    'Contractor_Billing': [
        'A contractor invoice was approved after timesheet verification',
        'Contractor payments were released on the net payment cycle',
        'A staffing agency invoice was disputed for extra hours',
        'Withholding tax was applied to a contractor payment',
        "A contractor's rate card was updated for the new year",
        'Procurement raised a purchase order for outside developers',
        'A late timesheet delayed a contractor payment',
        'A vendor invoice for consulting hours was queued for review',
        'Contractor expenses were reimbursed against receipts',
        "Contractor K214's invoice was approved after timesheet verification",
        "Contractor K230's rate card was updated after the engagement was extended",
        "A reimbursement of 800 dollars was approved for a contractor's travel",
        'A refund of 3000 dollars was issued to a vendor after a billing error',
    ],
    'Performance_Reviews': [
        'Mid-year review forms were opened for managers',
        'A calibration meeting was held for the engineering teams',
        'An employee requested feedback after a rating dispute',
        'Review ratings were locked after calibration',
        'A performance improvement plan was drafted for a struggling team member',
        'Peer feedback was collected for the product group',
        'Managers were reminded to complete review comments',
        'A self-assessment template was updated for the year',
        'Review outcomes were communicated in one-to-one meetings',
        'Employee E1021 received a strong mid-year rating after calibration',
        'Manager M09 completed review comments for all direct reports',
        'Employee E1066 was placed on a performance improvement plan',
        'Q3 appraisal ratings were frozen for the engineering teams',
    ],
    'Promotions': [
        'A promotion list was submitted for leadership approval',
        'Compensation bands were reviewed for the technology roles',
        'A salary revision letter was issued after a promotion',
        'A lateral move was approved between two departments',
        'The promotion committee deferred a case for more evidence',
        'An increment budget was allocated to each department',
        'A retention bonus was offered to a critical employee',
        'A role change was announced with a new reporting line',
        'Equity grants were scheduled for the next vesting date',
        'Employee E1034 was recommended for promotion by the committee',
        'Manager M05 submitted the promotion list for the design team',
        'Compensation bands for Grade L3 were revised for the next year',
    ],
    'Leave_Attendance': [
        'A parental leave request was approved by the manager',
        'Attendance records were corrected after a system outage',
        'A long weekend holiday was announced for all offices',
        'An employee applied for unpaid leave for personal reasons',
        'Leave balances were carried forward into the new year',
        'A comp-off request was approved for weekend work',
        'Biometric attendance was paused during maintenance',
        'Sick leave certificates were collected by HR',
        'A leave policy reminder was sent before the festival season',
        "Employee E1052's parental leave request was approved",
    ],
    'Workplace_Facilities': [
        'A desk allocation plan was published for the new office',
        'The cafeteria menu was updated after employee feedback',
        'An air conditioning fault was reported in the meeting room',
        'Parking slots were reassigned for the new quarter',
        'Access cards were replaced for two departments',
        'The security desk logged a visitor badge mismatch',
        'A hybrid seating system was rolled out across offices',
        'Fire drill was conducted at the main office',
        'The shuttle service timing was changed for night shifts',
        'Floor 5 was allocated to the product team in the new seating plan',
        'A water leak on Floor 5 was reported to the facilities desk',
        'Employee E1047 reported noisy construction on Floor 8',
        'Employee E1066 requested a quiet desk on Floor 8',
    ],
    'Training': [
        'A leadership workshop was announced for new managers',
        'An online course on data privacy was made mandatory',
        'A coding bootcamp was scheduled for junior engineers',
        'Training budget was approved for external certifications',
        'A language class was offered to relocating employees',
        'Safety training was completed by the plant staff',
        'A mentoring programme was launched for first-time managers',
        'Training attendance was recorded in the learning portal',
        'A skills gap survey was circulated to all departments',
        'Employee E1052 completed the data privacy course before going on leave',
        'Manager M09 attended the leadership workshop for new managers',
        'A training grant of 800 dollars was approved for an external certification',
    ],
    'Employee_Relations': [
        'A grievance was filed about unequal shift allocation',
        'Employee relations mediated a dispute between two colleagues',
        'A town hall was organised to discuss the new policy',
        'Anonymous feedback was reviewed by the culture committee',
        'A harassment awareness session was held for all teams',
        'A union representative met with HR about working hours',
        'The ethics hotline received a report that was routed for review',
        'An employee engagement survey was launched across offices',
        'A team outing was approved to improve morale',
        'Case HRC-3307 was opened for a grievance about shift allocation',
        'Case HRC-3307 was closed after mediation and a written resolution',
        'Manager M14 attended the town hall on the new policy',
    ],
    'Offboarding': [
        'An exit interview was scheduled for a departing employee',
        'Final settlement was prepared for a resigned employee',
        'Access was revoked on the last working day',
        'A notice period buyout was approved by HR',
        'Company assets were collected from a leaving employee',
        'A relieving letter was issued after clearance',
        'An alumni network invitation was sent to leavers',
        'Knowledge transfer sessions were planned before a resignation took effect',
        'A rehire request was reviewed against past exit notes',
        'Employee E1047 resigned and a final settlement was prepared',
    ],
    'Compliance': [
        'Annual code of conduct acknowledgement was collected',
        'A labour law update was shared with HR business partners',
        'An audit requested records of overtime approvals',
        'Data retention rules for personnel files were revised',
        'A background check vendor was changed after a contract review',
        'Mandatory POSH committee membership was refreshed',
        'Minimum wage changes were reviewed for contract staff',
        'A policy exception was documented and approved',
        'Statutory filings for the quarter were submitted on time',
        'Contractor K201 completed the code of conduct acknowledgement',
        'Case HRC-2218 was opened for a conflict of interest disclosure',
        'Q3 appraisal forms were sent to all managers',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. The pattern registry (the answer key)

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Employee',
        "regex": r"\bEmployee\s+(E\d{4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Employee E1034',
        "why": 'An employee ID names one person: the promotion recommendation and the payroll update for E1034 are about the same employee.',
    },
    {
        "name": 'Contractor',
        "regex": r"\bContractor\s+(K\d{3})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Contractor K214',
        "why": 'A contractor ID names one engagement; the onboarding access and the approved invoice for K214 are about the same contractor.',
    },
    {
        "name": 'Manager',
        "regex": r"\bManager\s+(M\d{2})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Manager M09',
        "why": 'A manager ID names one manager; the review comments and the leadership workshop for M09 are about the same person.',
    },
    {
        "name": 'Case',
        "regex": r"\b(HRC-\d{3,4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'HRC-3307',
        "why": 'An HR case number names one case from opening to resolution.',
    },
]

REJECTED_CANDIDATES = [
    {"candidate": 'Floor numbers (Floor 5, Floor 8)', "kind": 2, "why_rejected": "A floor is a place shared by many people and events: Employee E1047's construction complaint and E1066's quiet-desk request both mention Floor 8 but are unrelated."},
    {"candidate": 'Grade (Grade L3)', "kind": 2, "why_rejected": 'A grade is a level held by many roles and people; a band revision and a job opening at the same grade are separate events.'},
    {"candidate": 'Amounts (800 dollars, 3000 dollars)', "kind": 2, "why_rejected": 'Equal amounts can appear in unrelated records by coincidence: a 3000-dollar relocation allowance and an unrelated 3000-dollar vendor refund.'},
    {"candidate": 'Appraisal period (Q3 appraisal)', "kind": 2, "why_rejected": 'An appraisal period is a time window shared by many unrelated records.'},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 6. Build the graph

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

run_write("MATCH (n) DETACH DELETE n")

# One :Event node per record
event_rows = [{"id": f"R{i}", "text": t, "category": c}
              for i, (t, c) in enumerate(zip(rec_texts, rec_labels))]
run_write("""
    UNWIND $rows AS row
    CREATE (e:Event {id: row.id, text: row.text, category: row.category})
""", rows=event_rows)

# One entity node per distinct identifier; records naming it are linked through it
by_label = {}
for i, text in enumerate(rec_texts):
    for name, ident in extract_entities(text):
        by_label.setdefault(name, []).append({"eid": f"R{i}", "xid": ident})

for name, rows in by_label.items():       # name was validated: letters/digits/underscore only
    run_write(f"""
        UNWIND $rows AS r
        MATCH (e:Event {{id: r.eid}})
        MERGE (x:{name} {{id: r.xid}})
        MERGE (e)-[:MENTIONS]->(x)
    """, rows=rows)

print("Graph built from your registry:")
for name, rows in by_label.items():
    ids = {}
    for r in rows:
        ids.setdefault(r["xid"], 0)
        ids[r["xid"]] += 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {len(rows):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

## 7. Grounded answer

In [ ]:
def graph_check(id_a, id_b):
    """Asks Neo4j whether a path connects two records, and through which entities."""
    rows = run_read("""
        MATCH (a:Event {id: $a}), (b:Event {id: $b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN p IS NULL AS disconnected,
               [n IN nodes(p) WHERE NOT n:Event | labels(n)[0] + ' ' + n.id] AS via
    """, a=id_a, b=id_b)
    return rows[0] if rows else None


GROUNDED_PROMPT = """You answer questions about workforce operations (hiring, payroll, reviews, contractors, facilities) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Amounts, floor numbers, grades, appraisal periods, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 8. Question packs

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': 'Was the bonus payout released together with the contractor payments?',
  'trap_records': ['A bonus payout was scheduled with the next salary run',
                   'Contractor payments were released on the net payment cycle'],
  'correct_answer_must_state': 'No stated connection: a bonus payout and contractor payments are '
                               'separate records about pay.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (money)'},
 {'id': 'P2',
  'question': 'Is the revised offer letter the one whose onboarding paperwork was completed before '
              'the first day?',
  'trap_records': ['An offer letter was revised after the salary band was updated',
                   'Onboarding paperwork was completed before the first day'],
  'correct_answer_must_state': 'No evidence these records name the same person; they are separate '
                               'records.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (hiring)'},
 {'id': 'P3',
  'question': 'Did locking the review ratings cause the promotion committee to defer the case?',
  'trap_records': ['Review ratings were locked after calibration',
                   'The promotion committee deferred a case for more evidence'],
  'correct_answer_must_state': 'No stated connection: the deferral cites more evidence, and '
                               'nothing ties it to the ratings lock.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (review)'},
 {'id': 'P4',
  'question': 'Was the shuttle timing changed because of the long weekend holiday?',
  'trap_records': ['A long weekend holiday was announced for all offices',
                   'The shuttle service timing was changed for night shifts'],
  'correct_answer_must_state': 'No stated connection: the shuttle change concerns night shifts and '
                               'names no holiday.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (schedule)'},
 {'id': 'P5',
  'question': 'Did the new seating plan for the product team cause the water leak on Floor 5?',
  'trap_records': ['Floor 5 was allocated to the product team in the new seating plan',
                   'A water leak on Floor 5 was reported to the facilities desk'],
  'correct_answer_must_state': 'No stated connection: both mention Floor 5, but a floor is a place '
                               'shared by many events, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: floor'},
 {'id': 'P6',
  'question': 'Was the Grade L3 role opened because the compensation bands for Grade L3 were '
              'revised?',
  'trap_records': ['A Grade L3 role was opened for a backend developer',
                   'Compensation bands for Grade L3 were revised for the next year'],
  'correct_answer_must_state': 'No stated connection: both mention Grade L3, a level shared by '
                               'many roles, not one event.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: grade'},
 {'id': 'P7',
  'question': "Was the 800 dollar training grant the same payment as the contractor's travel "
              'reimbursement?',
  'trap_records': ['A training grant of 800 dollars was approved for an external certification',
                   "A reimbursement of 800 dollars was approved for a contractor's travel"],
  'correct_answer_must_state': 'No stated connection: the amounts match by coincidence, and the '
                               'purposes differ.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: amount'},
 {'id': 'P8',
  'question': 'Was the background check vendor changed because the coding bootcamp was scheduled?',
  'trap_records': ['A background check vendor was changed after a contract review',
                   'A coding bootcamp was scheduled for junior engineers'],
  'correct_answer_must_state': 'No stated connection between a vendor change and a bootcamp; any '
                               'causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Was Employee E1034 recommended for promotion, and did the new salary appear in '
              'payroll?',
  'shared_identifier': 'Employee E1034',
  'shared_records': ['Employee E1034 was recommended for promotion by the committee',
                     "Employee E1034's new salary was reflected in the payroll run"],
  'correct_answer_must_state': 'Yes -- both records name Employee E1034.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Did Employee E1052 complete the data privacy course and also have a parental leave '
              'request approved?',
  'shared_identifier': 'Employee E1052',
  'shared_records': ['Employee E1052 completed the data privacy course before going on leave',
                     "Employee E1052's parental leave request was approved"],
  'correct_answer_must_state': 'Yes -- both records name Employee E1052.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Was Contractor K214 issued access before starting, and was their invoice later '
              'approved?',
  'shared_identifier': 'Contractor K214',
  'shared_records': ['Contractor K214 was issued a laptop and access card before starting',
                     "Contractor K214's invoice was approved after timesheet verification"],
  'correct_answer_must_state': 'Yes -- both records name Contractor K214.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
HIDDEN_ADVERSARIAL = [{'id': 'H1',
  'question': 'Was the payroll overpayment correction linked to the disputed staffing agency '
              'invoice?',
  'trap_records': ['Payroll corrected an overpayment from the previous cycle',
                   'A staffing agency invoice was disputed for extra hours'],
  'correct_answer_must_state': 'No stated connection: a payroll correction and an agency invoice '
                               'dispute are independent records.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H2',
  'question': 'Was the retention bonus offered because the candidate declined the offer after a '
              'counter?',
  'trap_records': ['A candidate declined an offer after a counter from another company',
                   'A retention bonus was offered to a critical employee'],
  'correct_answer_must_state': 'No stated connection: nothing names the same person or role in '
                               'both records.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H3',
  'question': 'Did the attendance correction resolve the grievance about unequal shift allocation?',
  'trap_records': ['A grievance was filed about unequal shift allocation',
                   'Attendance records were corrected after a system outage'],
  'correct_answer_must_state': 'No stated connection: the correction followed a system outage and '
                               'names no case or employee.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H4',
  'question': 'Was the harassment awareness session part of the safety training completed by plant '
              'staff?',
  'trap_records': ['A harassment awareness session was held for all teams',
                   'Safety training was completed by the plant staff'],
  'correct_answer_must_state': 'No stated connection: two different training-style events, no '
                               'shared session or person.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H5',
  'question': 'Was the mentoring programme launched because of the planned knowledge transfer '
              'sessions?',
  'trap_records': ['Knowledge transfer sessions were planned before a resignation took effect',
                   'A mentoring programme was launched for first-time managers'],
  'correct_answer_must_state': 'No stated connection: one concerns a resignation, the other '
                               'first-time managers.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H6',
  'question': 'Did the cafeteria menu update come from the revised data retention rules for '
              'personnel files?',
  'trap_records': ['The cafeteria menu was updated after employee feedback',
                   'Data retention rules for personnel files were revised'],
  'correct_answer_must_state': 'No connection: a facilities update and a compliance revision do '
                               'not share vocabulary; any causal story is fabricated.',
  'expect_connected': False,
  'kind': None},
 {'id': 'H7',
  'question': "Did Employee E1066's request for a quiet desk on Floor 8 come from the construction "
              'noise Employee E1047 reported?',
  'trap_records': ['Employee E1047 reported noisy construction on Floor 8',
                   'Employee E1066 requested a quiet desk on Floor 8'],
  'correct_answer_must_state': 'No -- two different employees (E1047, E1066) mention Floor 8, but '
                               'a floor is a place reused by many events, not an event identifier. '
                               'Linking by floor would wrongly connect them.',
  'expect_connected': False,
  'kind2_trap': 'Floor (Floor 8)'},
 {'id': 'H8',
  'question': "Was the 3000 dollar vendor refund related to Employee E1021's relocation allowance?",
  'trap_records': ["Employee E1021's relocation allowance of 3000 dollars was approved",
                   'A refund of 3000 dollars was issued to a vendor after a billing error'],
  'correct_answer_must_state': "Employee E1021's 3000-dollar allowance is recorded, and a "
                               '3000-dollar vendor refund exists, but the refund names no '
                               'employee. The matching amount is incidental, not an event '
                               'identifier.',
  'expect_connected': False,
  'kind2_trap': 'Amount (3000 dollars)'},
 {'id': 'H9',
  'question': 'Was hiring for the support team paused because the Q3 appraisal ratings were frozen '
              'for engineering?',
  'trap_records': ['Hiring for the support team was paused until the Q3 appraisal closed',
                   'Q3 appraisal ratings were frozen for the engineering teams'],
  'correct_answer_must_state': 'No stated connection: both mention the Q3 appraisal, a period '
                               'shared by many records, not one event.',
  'expect_connected': False,
  'kind2_trap': 'Appraisal period (Q3 appraisal)'}]

HIDDEN_POSITIVE = [{'id': 'HP1',
  'question': 'Was grievance case HRC-3307 resolved after it was opened?',
  'shared_identifier': 'HRC-3307',
  'shared_records': ['Case HRC-3307 was opened for a grievance about shift allocation',
                     'Case HRC-3307 was closed after mediation and a written resolution'],
  'correct_answer_must_state': 'Yes -- both records carry case HRC-3307.',
  'expect_connected': True},
 {'id': 'HP2',
  'question': 'Did Employee E1021 join through a referral and also receive a strong mid-year '
              'rating?',
  'shared_identifier': 'Employee E1021',
  'shared_records': ['Employee E1021 joined the finance team through an employee referral',
                     'Employee E1021 received a strong mid-year rating after calibration'],
  'correct_answer_must_state': 'Yes -- both records name Employee E1021.',
  'expect_connected': True},
 {'id': 'HP3',
  'question': 'Did Manager M09 complete review comments and also attend the leadership workshop?',
  'shared_identifier': 'Manager M09',
  'shared_records': ['Manager M09 completed review comments for all direct reports',
                     'Manager M09 attended the leadership workshop for new managers'],
  'correct_answer_must_state': 'Yes -- both records name Manager M09.',
  'expect_connected': True},
 {'id': 'HP4',
  'question': 'Was Contractor K230 engaged through an agency, and was their rate card later '
              'updated?',
  'shared_identifier': 'Contractor K230',
  'shared_records': ['Contractor K230 was engaged through a staffing agency for six months',
                     "Contractor K230's rate card was updated after the engagement was extended"],
  'correct_answer_must_state': 'Yes -- both records name Contractor K230.',
  'expect_connected': True}]

print(f"Hidden pack: {len(HIDDEN_ADVERSARIAL)} adversarial + {len(HIDDEN_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### 8a. Public pack

In [ ]:
pub_adv = run_pack(PUBLIC_ADVERSARIAL)
pub_pos = run_pack(PUBLIC_POSITIVE)

### 8b. HIDDEN pack (judges)

Expect 9/9 adversarial NOT connected and 4/4 positive CONNECTED at the graph level. Read the LLM answers as well: they should say "no recorded link" for adversarial questions, and affirm and name the shared entity for positives. Where baseline and grounded disagree is the story to tell.

In [ ]:
hid_adv = run_pack(HIDDEN_ADVERSARIAL)
hid_pos = run_pack(HIDDEN_POSITIVE)

## 9. Teaching points

1. **Kind 1 vs Kind 2.** Kind 2 traps: Floor 8 (H7), the 3000-dollar amount (H8), the Q3 appraisal (H9); public pack has Floor 5, Grade L3 and 800 dollars. A team that adds those patterns fails exactly those questions.
2. **Coverage.** Hidden positives need Case (HP1), Manager (HP3) and Contractor (HP4) on top of Employee (HP2). A team that only adds the first obvious pattern fails the others.
3. **A baseline that fails only sometimes.** With a strict prompt the baseline can pass many of these questions; with the plain Week 2 prompt it is more likely to fabricate. Compare their answers on P5-P7 and H7-H9, and report what you actually see.
4. **Retrieval is separate.** If retrieval does not return both records, the graph check never sees them. Report that as a retrieval miss, not a grounding failure.